# Boston Crimes

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
pd.set_option('display.max_rows', None)
import datetime
from plotly.subplots import make_subplots

In [ ]:
data = pd.read_csv('crime.csv', encoding='latin')
data[:5]

In [ ]:
def treemap(categories, title, path, values):
    fig = px.treemap(categories, path=path, values=values, height=700,
                 title=title, color_discrete_sequence = px.colors.sequential.RdBu)
    fig.data[0].textinfo = 'label+text+value'
    fig.show()

In [ ]:
def histogram(data, path, color, title, xaxis: str, yaxis: str):
    fig = px.histogram(data, x=path, color=color)
    fig.update_layout(
        title_text=title,
        xaxis_title_text=xaxis, 
        yaxis_title_text=yaxis, 
        bargap=0.2, 
        bargroupgap=0.1
    )
    fig.show()

In [ ]:
def bar(categories, x, y, color, title, xlab, ylab):
    fig = px.bar(categories, x=x, y=y,
             color=color,
             height=400)
    fig.update_layout(
    title_text=title, 
    xaxis_title_text=xlab, 
    yaxis_title_text=ylab,
    bargap=0.2, 
    bargroupgap=0.1
    )
    fig.show()

In [ ]:
# Number of crimes per category
Number_crimes = data['OFFENSE_CODE_GROUP'].value_counts()
values = Number_crimes.values
categories = pd.DataFrame(data=Number_crimes.index, columns=["OFFENSE_CODE_GROUP"])
categories['values'] = values
categories[:5]

In [ ]:
treemap(categories,'Major Crimes in Boston',['OFFENSE_CODE_GROUP'],categories['values'])

In [ ]:
histogram(data,"OFFENSE_CODE_GROUP","OFFENSE_CODE_GROUP",'Major Crimes in Boston','Crime','Count')

In [ ]:
bar(categories,categories['OFFENSE_CODE_GROUP'][0:10],categories['values'][0:10]
    ,categories['OFFENSE_CODE_GROUP'][0:10],'Top 10 Major Crimes in Boston','Crime','Count')

In [ ]:
Number_crimes_year = data['YEAR'].value_counts()
years = pd.DataFrame(data=Number_crimes_year.index, columns=["YEAR"])
years['values'] = Number_crimes_year.values
years

In [ ]:
fig = px.pie(years, values='values', names='YEAR', color_discrete_sequence=px.colors.sequential.RdBu)
fig.show()

In [ ]:
histogram(data,"DISTRICT","MONTH",'Crime count per Month on each district', 'District', 'Crimes Count on each Day')

### Exercise 1
Use any plots for this exercise, can use different plots for different exercises.



1. What are the months with highest crime rate? Plot the data.


In [ ]:
series_crimes = data["MONTH"].value_counts().sort_index()
df_crimes = pd.DataFrame({'MONTH': series_crimes.index, 'COUNT': series_crimes.values})
bar(df_crimes, "MONTH", "COUNT", 'MONTH', 'Crime count per District on each Month', 'Month', 'Crimes Count on each Day')


2. What are the days of the week with highest crime rate? Plot the data.


In [ ]:
series_weekday_crimes = data['DAY_OF_WEEK'].value_counts()
df_crimes = pd.DataFrame({'DAY_OF_WEEK': series_weekday_crimes.index, 'COUNT': series_weekday_crimes.values})
bar(df_crimes, "DAY_OF_WEEK", "COUNT", 'DAY_OF_WEEK', 'Crime count on each Day', 'Day', 'Crimes Count on each Day')


3. What are the districts with highest crime rate? Plot the data.


In [ ]:
series_weekday_crimes = data['DISTRICT'].value_counts()
df_crimes = pd.DataFrame({'DISTRICT': series_weekday_crimes.index, 'COUNT': series_weekday_crimes.values})
bar(df_crimes, "DISTRICT", "COUNT", 'DISTRICT', 'Crime count on each DISTRICT', 'District', 'Crimes Count')


4. Plot the crime rate per district per day of week.

In [ ]:
histogram(data,"DISTRICT","DAY_OF_WEEK",'Crime count per Day on each district', 'District', 'Crimes Count')

## Map 

In [ ]:
%%capture
%pip install folium

In [ ]:
import folium
from folium import Choropleth, Circle, Marker
from folium.plugins import HeatMap, MarkerCluster


In [ ]:
# Create a map centered on Boston
m_1 = folium.Map(location=[42.32,-71.0589], tiles='openstreetmap', zoom_start=10)

# Display the map
m_1

In [ ]:
import pandas as pd
# Load the data
crimes = pd.read_csv("crime.csv", encoding='latin-1')
crimes

# Drop rows with missing locations
crimes.dropna(subset=['Lat', 'Long', 'DISTRICT'], inplace=True)

# Focus on major crimes
crimes = crimes[crimes.OFFENSE_CODE_GROUP.isin([
    'Larceny', 'Auto Theft', 'Robbery', 'Larceny From Motor Vehicle', 'Residential Burglary',
    'Simple Assault', 'Harassment', 'Ballistics', 'Aggravated Assault', 'Other Burglary', 
    'Arson', 'Commercial Burglary', 'HOME INVASION', 'Homicide', 'Criminal Harassment', 
    'Manslaughter'])]

# Print the first five rows of the table
crimes.head()

In [ ]:
daytime_robberies = crimes[((crimes.OFFENSE_CODE_GROUP == 'Robbery') & \
                            (crimes.HOUR.isin(range(9,18))))]

In [ ]:
# Create a map with markers
m_2 = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=13)

# Add daytime robberies to the map
for idx, row in daytime_robberies.iterrows():
    Marker([row['Lat'], row['Long']]).add_to(m_2)

# Display the map - All daytime robberies
m_2

In [ ]:
# Marker clusters
import math
# Create the map
m_3 = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=13)

# Add points to the map
mc = MarkerCluster()
for idx, row in daytime_robberies.iterrows():
    if not math.isnan(row['Long']) and not math.isnan(row['Lat']):
        mc.add_child(Marker([row['Lat'], row['Long']]))
m_3.add_child(mc)

# Display the map
m_3

In [ ]:
# Create a base map
m_4 = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=12)

def color_producer(val):
    if val <= 12:
        return 'forestgreen'
    else:
        return 'darkred'

# Add a bubble to the base map
for i in range(0,len(daytime_robberies)):
    Circle(
        location=[daytime_robberies.iloc[i]['Lat'], daytime_robberies.iloc[i]['Long']],
        radius=20,
        color=color_producer(daytime_robberies.iloc[i]['HOUR'])).add_to(m_4)

# Display the map
m_4

### Exercise 2



1. Plot all of the crimes that occurred in district D14 during 2018


In [ ]:
d14_crimes = crimes[(crimes['DISTRICT'] == 'D14') & (crimes['YEAR'] == 2018)]

In [ ]:
m_21 = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=13)

# Add daytime robberies to the map
for idx, row in d14_crimes.iterrows():
    Marker([row['Lat'], row['Long']]).add_to(m_21)
    
m_21


2. Plot all of the robberies that occurred on Friday evenings (after 18:00 o'clock).


In [ ]:
friday_evening_robberies = crimes[((crimes.OFFENSE_CODE_GROUP == 'Robbery') & (crimes.DAY_OF_WEEK == 'Friday') & (crimes.HOUR.isin(range(18,24))))]

In [ ]:
m_22 = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=13)

# Add daytime robberies to the map
for idx, row in friday_evening_robberies.iterrows():
    Marker([row['Lat'], row['Long']]).add_to(m_22)
    
m_22


3. Plot all of the robberies that occured on the 4th of July.


In [ ]:
patriot_crimes = crimes[crimes['OCCURRED_ON_DATE'].str.match(r'^[0-9]{4}-07-04')]

In [ ]:
m_23 = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=13)

# Add daytime robberies to the map
for idx, row in patriot_crimes.iterrows():
    Marker([row['Lat'], row['Long']]).add_to(m_23)
    
m_23

4. Plot all of the crimes that occurred on the 1st of January.

In [ ]:
new_year_crimes = crimes[crimes['OCCURRED_ON_DATE'].str.match(r'^[0-9]{4}-01-01')]

In [ ]:
m_24 = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=13)

# Add daytime robberies to the map
for idx, row in new_year_crimes.iterrows():
    Marker([row['Lat'], row['Long']]).add_to(m_24)
    
m_24

5. Count (separately) the number of crimes that occured on the 1st of January, December 31st, 4th of July and May 12th. Compare the 4 results.




In [ ]:
jan_01st_crime_count = len(crimes[crimes['OCCURRED_ON_DATE'].str.match(r'^[0-9]{4}-01-01')])
dec_31st_crime_count = len(crimes[crimes['OCCURRED_ON_DATE'].str.match(r'^[0-9]{4}-12-31')])
jul_04th_crime_count = len(crimes[crimes['OCCURRED_ON_DATE'].str.match(r'^[0-9]{4}-07-04')])
may_12th_crime_count = len(crimes[crimes['OCCURRED_ON_DATE'].str.match(r'^[0-9]{4}-05-12')])

event_crimes_df = pd.DataFrame({'Date': ['Jan 1st', 'Dec 31st', 'July 4th', 'May 12th'],
                                'Count': [jan_01st_crime_count, dec_31st_crime_count, jul_04th_crime_count, may_12th_crime_count]}) 

bar(event_crimes_df, 'Date', 'Count', 'Date', 'Event Crimes', 'Date', 'Count')

In [ ]:
import pandas as pd
# Load the data
earthquakes = pd.read_csv("database.csv", encoding='latin-1')
earthquakes.head()


In [ ]:
# Create a map
m_2 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=3)

cnt = 0

# Add first 100 earthquakes to the map
for idx, row in earthquakes.iterrows():
    Marker([row['Latitude'], row['Longitude']]).add_to(m_2)

    cnt +=1 
    if cnt == 100:
        break

# Display the map
m_2

### Exercise 3



1. Plot all of the earthquakes that occurred between 41 and 42 degrees latitude.


In [ ]:
latitude_41_42_earthquakes = earthquakes[(earthquakes['Latitude'] >= 41) & (earthquakes['Latitude'] <= 42)]

In [ ]:
m_31 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=3)

for idx, row in latitude_41_42_earthquakes.iterrows():
    Marker([row['Latitude'], row['Longitude']]).add_to(m_31)

m_31


2. Plot all of the earthquakes that occurred in Boston (or at max 1 degree latitude and 1 degree longitude away from it). How many earthquakes did Boston suffer?
   


In [ ]:
# Boston Coordinates: 42.32,-71.0589
# accuracy of 1 degree
boston_earthquakes = earthquakes[(earthquakes['Latitude'] >= 41.32) & (earthquakes['Latitude'] <= 43.32) & (earthquakes['Longitude'] >= -72.0589) & (earthquakes['Longitude'] <= -70.0589)]

boston_earthquakes.head()

3. Plot all of the earthquakes that were of magnitude greater than 7 using circles on the map. Use the magnitude to increase the circle radius (higher magnitude - bigger circle.

In [ ]:
magnitude_7_earthquakes = earthquakes[(earthquakes['Magnitude'] >= 7)]

In [ ]:
m_33 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=3)

def map_range_to_range(val, in_min, in_max, out_min, out_max):
    return (val - in_min) * (out_max - out_min) / (in_max - in_min) + out_min

# Add a bubble to the base map
for i in range(0,len(magnitude_7_earthquakes)):
    Circle(
        location=[magnitude_7_earthquakes.iloc[i]['Latitude'], magnitude_7_earthquakes.iloc[i]['Longitude']],
        radius=map_range_to_range(magnitude_7_earthquakes.iloc[i]['Magnitude'], 7, 10, 5000, 200000)
    ).add_to(m_33)

# Display the map
m_33

### Exercise 4


In [ ]:
import pandas as pd
# Load the data
cities = pd.read_csv("worldcities.csv", encoding='latin-1')
cities[:5]


1. Plot all of cities in Romania using circles and increase the circle radius based on population.


In [ ]:
romanian_cities = cities[cities['country'] == 'Romania']

In [ ]:
m_41 = folium.Map(location=[46, 25], tiles='cartodbpositron', zoom_start=8)

# Add a bubble to the base map
for i in range(0,len(romanian_cities)):
    Circle(
        location=[romanian_cities.iloc[i]['lat'], romanian_cities.iloc[i]['lng']],
        radius=romanian_cities.iloc[i]['population']/100
    ).add_to(m_41)

# Display the map
m_41


2. Plot all of the earthquakes that took place at max 5 degrees latitude and longitude away from Bucharest. (lat_eq + long_eq - lat_buch - long_buch > 5)


In [ ]:
bucharest_earthquakes = earthquakes[((earthquakes['Latitude'] - 44.42)**2 + (earthquakes['Longitude'] - 26.1)**2) < 25]

In [ ]:
m_42 = folium.Map(location=[46, 25], tiles='cartodbpositron', zoom_start=8)

# Add a bubble to the base map
for i in range(0,len(bucharest_earthquakes)):
    Circle(
        location=[bucharest_earthquakes.iloc[i]['Latitude'], bucharest_earthquakes.iloc[i]['Longitude']],
        radius=200
    ).add_to(m_42)

# Display the map
m_42

### Exercise 5
Create two useful visualizations (graph plots and/or map plots) by making use of information from at least two of the datasets provided in the laboratory.

#### **Internet Speeds**

In [ ]:
internet_speeds = pd.read_csv("extra_data/internet_speed.csv", encoding='latin-1')

internet_speeds.head(5)

In [ ]:
import re

def tile_to_coords(tile):
    coords = re.findall(r'[-+]?(?:\d*\.\d+|\d+)', tile)
    
    return [float(coords[1]), float(coords[0])]

m_51 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=3)
  
for idx, row in internet_speeds.iterrows():
    coords = tile_to_coords(row['tile'])
    
    Circle(
        location=[coords[0], coords[1]],
        radius=row['avg_d_kbps'],
        color='red'
    ).add_to(m_51)
    Circle(
        location=[coords[0], coords[1]],
        radius=row['avg_u_kbps'],
        color='blue'
    ).add_to(m_51)

# Display the map
m_51

#### **Mischelin Resatarants**

In [ ]:
michelin_restaurants_1_star = pd.read_csv("extra_data/one-star-michelin-restaurants.csv", encoding='latin-1')
michelin_restaurants_2_star = pd.read_csv("extra_data/two-stars-michelin-restaurants.csv", encoding='latin-1')
michelin_restaurants_3_star = pd.read_csv("extra_data/three-stars-michelin-restaurants.csv", encoding='latin-1')

michelin_restaurants_1_star.head()

In [ ]:
m_52 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=3)

def plot_restaurants(df, color):
    marker_cluster = MarkerCluster().add_to(m_52)
    for _, row in df.iterrows():
        Marker(
            location=[row['latitude'], row['longitude']],
        ).add_to(marker_cluster)
        
plot_restaurants(michelin_restaurants_1_star, 'red')
plot_restaurants(michelin_restaurants_2_star, 'blue')
plot_restaurants(michelin_restaurants_3_star, 'green')

m_52

#### **Tectonic Plates**

In [ ]:
tectonic_plates = pd.read_csv("extra_data/tectonic_plates.csv", encoding='latin-1')

tectonic_plates.head()

In [ ]:
def df_to_coords(df):
    coords = []
    avg_lat = df['lat'].mean()
    avg_lon = df['lon'].mean()
    
    for _, row in df.iterrows():
        coords.append([row['lon'], row['lat']])

    return pd.Series({
        'avg_lat': avg_lat,
        'avg_lon': avg_lon,
        'coords_len': len(coords),
        'coords': coords
    })

plate_df = tectonic_plates.groupby(['plate']).apply(df_to_coords).reset_index()

plate_df.head()

In [ ]:
def coords_to_geojson(plate: str, coords): 
    return {
        "type": "Feature",
        "id": plate,
        "geometry": {
            "type": "Polygon",
            "coordinates": [
                coords
            ]
        }
    }

coords_geo_json = [coords_to_geojson(row['plate'], row['coords']) for _, row in plate_df.iterrows()]

geo_json = {
  "type": "FeatureCollection",
  "features": coords_geo_json
}

In [ ]:
choro_data = plate_df[['plate', 'coords_len']]

In [ ]:
m_53 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=2)

Choropleth(
    geo_data=geo_json,
    data=choro_data,
    columns=["plate", "coords_len"],
    key_on="feature.id",
    fill_color="BuPu",
    fill_opacity=0.7,
    line_opacity=1,
    legend_name="Number of Coords",
).add_to(m_53)

m_53

#### **World Capitals**

In [ ]:
world_capitals = pd.read_csv("extra_data/world_capitals.csv", encoding='latin-1')

world_capitals.head()

In [ ]:
m_54 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=3)
  
for _, row in world_capitals.iterrows():
    Marker(
        location=[row['CapitalLatitude'], row['CapitalLongitude']],
        popup=f'{row["CapitalName"]}, {row["CountryName"]}',
    ).add_to(m_54)

# Display the map
m_54

#### **Volcanic Eruptions**

In [ ]:
volcanos = pd.read_csv("extra_data/volcano.csv", encoding='latin-1')
volcanos.drop(['minor_rock_1', 'minor_rock_2', 'minor_rock_3', 'minor_rock_4', 'minor_rock_5', 'major_rock_1', 'major_rock_2', 'major_rock_3', 'major_rock_4', 'major_rock_5'], axis=1, inplace=True)
volcanos.head()

In [ ]:
eruptions = pd.read_csv("extra_data/eruptions.csv", encoding='latin-1') 
eruptions.head()

In [ ]:
eruptions_21st_centruy = eruptions[eruptions['start_year'] >= 2000]

In [ ]:
m_55 = folium.Map(location=[0, 0], tiles='cartodbpositron', zoom_start=3)
  
for _, row in volcanos.iterrows():
    Circle(
        location=[row['latitude'], row['longitude']],
        radius=20
    ).add_to(m_55)

for _, row in eruptions_21st_centruy.iterrows():
    Circle(
        location=[row['latitude'], row['longitude']],
        radius=40,
        color='red'
    ).add_to(m_55)

# Display the map
m_55